# Day 9: Documents, Chunks and Vectors

LangChain in 10 Days · KGP Talkie · Laxmi Kant Tiwari · kgptalkie.com

**Today:** cut a document into chunks, turn a chunk into numbers, and find the right one by meaning, not keyword. Do the arithmetic by hand once, then hand it to a real vector database, and end on a failure Day 10 exists to fix.

> Sections 9 onward need Qdrant running. From the project folder: `docker compose up -d`

In [1]:
import warnings

warnings.filterwarnings("ignore")

Silences a `langchain-community` deprecation warning further down. Nothing is broken.

## 1. Load the document we will search

In [2]:
from pathlib import Path

text = Path("data/baggage-policy.md").read_text(encoding="utf-8")

len(text)

495

In [3]:
print(text)

# Zephyr Airlines Baggage Policy 2026

## Cabin baggage
Every passenger may carry one cabin bag up to 7 kg and one personal item.
Cabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.

## Checked baggage
Economy fare includes 15 kg of checked baggage. Business fare includes 30 kg.
Excess baggage is charged at 600 rupees per kilogram.

## Delayed baggage
Report delayed baggage at the arrival desk within 24 hours.
Zephyr Airlines pays 2000 rupees per day for up to five days.



## 2. Split the document into chunks

A real handbook won't fit in one prompt, and you'd pay for all of it to answer what one paragraph covers. So it gets cut up.

In [4]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

splitter = RecursiveCharacterTextSplitter(chunk_size=250, chunk_overlap=40)

chunks = splitter.split_text(text)

chunks

['# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.',
 '## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.',
 '## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.']

- `chunk_size` is roughly how big each piece is, in characters.
- `chunk_overlap` repeats a little of the previous piece, so a boundary sentence isn't lost to both sides.

The splitter tried blank lines first, then newlines, then sentences, chopping mid-word only as a last resort.

In [5]:
splitter._separators

['\n\n', '\n', ' ', '']

That list, in order, is the whole "recursive" idea: work down it until the pieces are small enough.

## 3. How chunk size changes the result

Try three sizes on the same document.

In [6]:
for size in (100, 250, 1000):
    pieces = RecursiveCharacterTextSplitter(
        chunk_size=size, chunk_overlap=0
    ).split_text(text)

    print(str(size) + " chars -> " + str(len(pieces)) + " chunks")

100 chars -> 7 chunks
250 chars -> 3 chunks
1000 chars -> 1 chunks


In [7]:
RecursiveCharacterTextSplitter(chunk_size=100, chunk_overlap=0).split_text(text)[:4]

['# Zephyr Airlines Baggage Policy 2026',
 '## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.',
 'Cabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.',
 '## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.']

At 100 characters the pieces are fragments, a heading separates from the rule underneath it, retrieving one gives you neither.

In [8]:
RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=0).split_text(text)[0]

'# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.\n\n## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.\n\n## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.'

At 1000 the whole document is nearly one chunk. Retrieval returns everything, same as no retrieval, and you pay for the lot.

| Chunks too small | Chunks too large |
|---|---|
| Fragments lose context | One chunk covers many topics |
| Answer split across pieces | Matching part gets diluted |
| More chunks to search | You pay for text you didn't need |

No correct number. 250 to 800 characters is a reasonable start, then test against real questions.

## 4. Turn text into an embedding vector

This is an **embedding**. First run downloads a small model, ~130 MB, then it's instant and offline.

In [9]:
from langchain_community.embeddings import FastEmbedEmbeddings

embedder = FastEmbedEmbeddings(model_name="BAAI/bge-small-en-v1.5")

vector = embedder.embed_query("How much baggage can I check in?")

len(vector)

384

384 numbers, what a sentence looks like to a computer. You'll never read these, what matters: **two sentences with similar meaning get similar numbers**, even sharing no words.

## 5. Check that similar meaning gives similar vectors

In [10]:
import math

def dot(a, b):
    return sum(x * y for x, y in zip(a, b))

luggage = embedder.embed_query("How much luggage can I bring?")
baggage = embedder.embed_query("What is the checked baggage allowance?")
paris = embedder.embed_query("The capital of France is Paris.")

In [11]:
# paris

In [12]:
round(math.sqrt(dot(luggage, luggage)), 4)

1.0

Length 1: these vectors are normalised, so the dot product of two **is** the cosine of the angle between them. One line of arithmetic gives you similarity.

In [13]:
round(dot(luggage, baggage), 4)

0.7374

In [14]:
round(dot(luggage, paris), 4)

0.4594

The two baggage questions share almost no words, "luggage"/"bring" against "checked baggage"/"allowance", yet score far higher than the Paris sentence. That's semantic search.

### Why a similarity score is not a percentage

The unrelated pair didn't score near zero. Similarity scores sit in a narrow, high baseline band, **not a percentage of relevance**. A score is only meaningful compared to other scores for the same question, which is why you rank rather than threshold.

## 6. Embed all the chunks at once

In [15]:
chunks

['# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.',
 '## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.',
 '## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.']

In [16]:
chunk_vectors = embedder.embed_documents(chunks)

len(chunk_vectors)

3

In [17]:
# chunk_vectors

In [18]:
chunks

['# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.',
 '## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.',
 '## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.']

## 7. Search by hand, in pure Python

In [19]:
question = "How much luggage is included with an economy ticket?"

q_vector = embedder.embed_query(question)

scores = [dot(q_vector, c) for c in chunk_vectors]

In [20]:
scores

[0.7381997385637132, 0.8111264396814926, 0.7173213524552918]

In [21]:


best = scores.index(max(scores))

chunks[best]

'## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.'

The question says **luggage** and **economy ticket**, the document says **checked baggage** and **Economy fare**. Not one important word matches, yet it found the right paragraph. A keyword search would have returned nothing.

In [22]:
ranked = sorted(zip(scores, chunks), reverse=True)

for score, chunk in ranked:
    print(str(round(score, 3)) + "  " + chunk[:70].replace("\n", " "))

0.811  ## Checked baggage Economy fare includes 15 kg of checked baggage. Bus
0.738  # Zephyr Airlines Baggage Policy 2026  ## Cabin baggage Every passenge
0.717  ## Delayed baggage Report delayed baggage at the arrival desk within 2


The top chunk is the answer, the others are near misses, scores close together: that narrow band again.

## 8. Store the vectors in Qdrant

Everything above was a Python list and a `for` loop, fine for four chunks, hopeless for four million where you'd compare against every vector every time.

A vector database does this with an index. Yours is running in Docker.

In [23]:
from qdrant_client import QdrantClient

client = QdrantClient(url="http://localhost:6333")

[c.name for c in client.get_collections().collections]

['airline_docs',
 'capstone_filings',
 'day09_baggage',
 'filings',
 'filings_10k',
 'filings_day21',
 'filings_day22',
 'filings_day24',
 'filings_day30',
 'filings_full',
 'filings_hybrid',
 'finance-rag-bootcamp',
 'finance-rag-ollama',
 'financial_docs',
 'financial_docs_dense']

If that raised, Qdrant is not running. Go to the project folder and run
`docker compose up -d`, then run the cell again.

### Wrap chunks in Document objects and load them

`Document` is the object LangChain passes around: some text, plus metadata you
choose.

In [24]:
chunks

['# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.',
 '## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.',
 '## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.']

In [25]:
from langchain_core.documents import Document
from langchain_qdrant import QdrantVectorStore

docs = [
    Document(page_content=c, metadata={"source": "baggage-policy.md", "chunk": i})
    for i, c in enumerate(chunks)
]

docs

[Document(metadata={'source': 'baggage-policy.md', 'chunk': 0}, page_content='# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.'),
 Document(metadata={'source': 'baggage-policy.md', 'chunk': 1}, page_content='## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.'),
 Document(metadata={'source': 'baggage-policy.md', 'chunk': 2}, page_content='## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.')]

In [26]:
docs[2].page_content

'## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.'

In [27]:
store = QdrantVectorStore.from_documents(
    docs,
    embedding=embedder,
    url="http://localhost:6333",
    collection_name="day09_baggage",
    force_recreate=True,
)

That embedded every chunk and sent it to the container. `force_recreate=True` wipes the collection first, so rerunning doesn't duplicate.

In [28]:
client.get_collection("day09_baggage").points_count

3

### Run a similarity search

In [29]:
question

'How much luggage is included with an economy ticket?'

In [30]:
store.similarity_search(question, k=2)

[Document(metadata={'source': 'baggage-policy.md', 'chunk': 1, '_id': 'c228aa2c-c030-4b51-aa3b-645d23eacbcd', '_collection_name': 'day09_baggage'}, page_content='## Checked baggage\nEconomy fare includes 15 kg of checked baggage. Business fare includes 30 kg.\nExcess baggage is charged at 600 rupees per kilogram.'),
 Document(metadata={'source': 'baggage-policy.md', 'chunk': 0, '_id': 'f3ca3442-791c-4bbc-bb4b-d165fd530c38', '_collection_name': 'day09_baggage'}, page_content='# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.')]

The same result as your hand-written loop, without the loop.

In [31]:
for doc, score in store.similarity_search_with_score(question, k=3):
    print(str(round(score, 3)) + "  " + doc.page_content[:60].replace("\n", " "))

0.811  ## Checked baggage Economy fare includes 15 kg of checked ba
0.738  # Zephyr Airlines Baggage Policy 2026  ## Cabin baggage Ever
0.717  ## Delayed baggage Report delayed baggage at the arrival des


Same ordering as by hand. Nothing magical happened, the database just did your arithmetic quickly, with an index.

### Filter a search by metadata

Metadata travels with each chunk, and you can filter on it.

In [32]:
store.similarity_search(question, k=2, filter={"must": [{"key": "metadata.chunk", "match": {"value": 0}}]})

[Document(metadata={'source': 'baggage-policy.md', 'chunk': 0, '_id': 'f3ca3442-791c-4bbc-bb4b-d165fd530c38', '_collection_name': 'day09_baggage'}, page_content='# Zephyr Airlines Baggage Policy 2026\n\n## Cabin baggage\nEvery passenger may carry one cabin bag up to 7 kg and one personal item.\nCabin bags larger than 55 x 35 x 25 cm must be checked in at the gate.')]

Restricting the search to a subset, this customer's documents, this year, this department, is the everyday reason to use a database over a list.

Vectors also outlive your kernel: restart Python and the collection is still in the container. The difference between an experiment and an application.

## 9. Where vector search fails: it always returns something

Ask this document something it does not contain.

In [33]:
store.similarity_search("What is the refund policy for cancelled flights?", k=1)

[Document(metadata={'source': 'baggage-policy.md', 'chunk': 2, '_id': '5da8caed-6d34-46a6-83e4-fca4e687e6cd', '_collection_name': 'day09_baggage'}, page_content='## Delayed baggage\nReport delayed baggage at the arrival desk within 24 hours.\nZephyr Airlines pays 2000 rupees per day for up to five days.')]

It returned a chunk. The document has no refund section, nothing to find, yet the search found something anyway: no error, no warning, no empty result.

**A vector search always returns its nearest neighbours.** It has no concept of "not in here". Ask for the nearest 3 and you get 3, every time, whatever you asked.

In [34]:
for doc, score in store.similarity_search_with_score("What is the refund policy for cancelled flights?", k=3):
    print(str(round(score, 3)) + "  " + doc.page_content[:60].replace("\n", " "))

0.614  ## Delayed baggage Report delayed baggage at the arrival des
0.566  ## Checked baggage Economy fare includes 15 kg of checked ba
0.528  # Zephyr Airlines Baggage Policy 2026  ## Cabin baggage Ever


Compare those scores to section 8: lower, but not obviously wrong, no cutoff would work across every question.

Hand that top chunk to a model with "answer using this context" and it writes a confident paragraph about refunds, assembled out of a baggage policy.

That's exactly how confident nonsense gets produced, and it's where tomorrow starts.